In [ ]:
# https://www.nyu.edu/about/news-publications/news/2024/april/how-do-birds-flock--researchers-do-the-math-to-reveal-previously.html

import pygame
import random
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Screen settings
WIDTH, HEIGHT = 800, 600
FPS = 60

# Boid settings
NUM_BOIDS = 60
MAX_SPEED = 4
K_NEIGHBORS = 2  # Number of nearest neighbors

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 100, 255)

class Boid:
    def __init__(self, x, y):
        self.position = pygame.Vector2(x, y)
        self.velocity = pygame.Vector2(random.uniform(-1, 1), random.uniform(-1, 1)).normalize() * MAX_SPEED
        self.acceleration = pygame.Vector2(0, 0)

    def update(self):
        # Update velocity and position
        self.velocity += self.acceleration
        if self.velocity.length() > MAX_SPEED:
            self.velocity.scale_to_length(MAX_SPEED)
        self.position += self.velocity
        self.acceleration = pygame.Vector2(0, 0)

        # Wrap around the screen
        if self.position.x > WIDTH:
            self.position.x = 0
        elif self.position.x < 0:
            self.position.x = WIDTH
        if self.position.y > HEIGHT:
            self.position.y = 0
        elif self.position.y < 0:
            self.position.y = HEIGHT

    def apply_behavior(self, neighbors):
        separation = self.separate(neighbors)
        alignment = self.align(neighbors)
        cohesion = self.cohere(neighbors)

        # Weigh the forces
        self.acceleration += separation
        self.acceleration += alignment
        self.acceleration += cohesion

    def separate(self, neighbors):
        steer = pygame.Vector2(0, 0)
        for other in neighbors:
            distance = self.position.distance_to(other.position)
            diff = self.position - other.position
            diff /= max(distance, 0.1)  # Avoid division by zero
            steer += diff
        if len(neighbors) > 0:
            steer /= len(neighbors)
        if steer.length() > 0:
            steer = steer.normalize() * MAX_SPEED - self.velocity
        return steer

    def align(self, neighbors):
        avg_velocity = pygame.Vector2(0, 0)
        for other in neighbors:
            avg_velocity += other.velocity
        if len(neighbors) > 0:
            avg_velocity /= len(neighbors)
            avg_velocity = avg_velocity.normalize() * MAX_SPEED - self.velocity
        return avg_velocity

    def cohere(self, neighbors):
        center_of_mass = pygame.Vector2(0, 0)
        for other in neighbors:
            center_of_mass += other.position
        if len(neighbors) > 0:
            center_of_mass /= len(neighbors)
            return (center_of_mass - self.position).normalize() * MAX_SPEED - self.velocity
        return pygame.Vector2(0, 0)

# Initialize Pygame
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("kNN Boids Simulation")
clock = pygame.time.Clock()

# Create boids
boids = [Boid(random.randint(0, WIDTH), random.randint(0, HEIGHT)) for _ in range(NUM_BOIDS)]

running = True
while running:
    screen.fill(BLACK)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Collect positions for kNN
    positions = np.array([[boid.position.x, boid.position.y] for boid in boids])
    nbrs = NearestNeighbors(n_neighbors=K_NEIGHBORS).fit(positions)
    _, indices = nbrs.kneighbors(positions)

    # Update and draw boids
    for i, boid in enumerate(boids):
        neighbors = [boids[j] for j in indices[i] if j != i]  # Exclude itself
        boid.apply_behavior(neighbors)
        boid.update()
        pygame.draw.circle(screen, BLUE, (int(boid.position.x), int(boid.position.y)), 5)

    pygame.display.flip()
    clock.tick(FPS)

pygame.quit()


pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
